In [0]:
from pyspark.sql import functions as F

silver_v2_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/silver/tlc_yellow_trips_v2/"
gold_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/gold/yellow_monthly/"

storage_account = "tlcyifanadelaide2026"

spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  "TxUTuvrZZJ9LMKhOK6GI81YaniXXrBw2TfnA08X9fgQ1EeQ4sS20rGZmaRjnWqEswZe4l8LMmHsa+AStSuqpyQ=="
)

df = spark.read.format("delta").load(silver_v2_path)

gold = (df
    .groupBy("pickup_year","pickup_month")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_amount_sum"),
        F.sum("tip_amount").alias("tip_amount_sum"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_duration_sec").alias("avg_duration_sec")
    )
)

# write parquet for Power BI
(gold.write.mode("overwrite").parquet(gold_path))